In [2]:
import pandas as pd
import numpy as np

# 파일 경로 지정
reviews_path = "../data/raw/reviews2.csv"
store_path = "../data/raw/ymj.csv"

# CSV 불러오기
reviews = pd.read_csv(reviews_path)
stores = pd.read_csv(store_path)

# reviews2 데이터의 store_id 종류 확인
num_unique_store_ids = reviews["store_id"].nunique()
print(f"reviews2 데이터의 고유 store_id 개수: {num_unique_store_ids}")

# 공통 store_id만 필터링
filtered_stores = stores[stores["store_id"].isin(reviews["store_id"])]

# 결과 확인
print(f"필터링 전 행 수: {len(stores)}")
print(f"필터링 후 행 수: {len(filtered_stores)}")

# 새 CSV로 저장 (선택사항)
filtered_stores.to_csv("../data/raw/ymj2.csv", index=False, encoding="utf-8-sig")

reviews2 데이터의 고유 store_id 개수: 125
필터링 전 행 수: 14300
필터링 후 행 수: 125


In [3]:
import pandas as pd

# 1. 파일 경로 지정
reviews_path = "../data/raw/reviews2.csv"
store_path = "../data/raw/ymj2.csv"  # 원본 ymj 파일 사용
output_path = "../data/raw/ymj3.csv"

# 2. CSV 파일 불러오기
try:
    reviews = pd.read_csv(reviews_path)
    stores = pd.read_csv(store_path)
except FileNotFoundError as e:
    print(f"오류: 파일을 찾을 수 없습니다. 경로를 확인해 주세요. - {e}")
    exit()

# 3. 'reviews' 데이터에서 업데이트할 정보(store_id, category) 추출
#    - 만약 'reviews'에 동일한 store_id에 여러 카테고리가 있다면, 첫 번째 값만 사용합니다.
category_updates = reviews[['store_id', 'category']].drop_duplicates(subset='store_id')

# 4. ★★★ 핵심 수정 사항 ★★★
#    'stores' 데이터와 컬럼명을 맞추기 위해 'category'를 'category_id'로 변경합니다.
#    이렇게 해야 .update()가 어떤 컬럼을 업데이트할지 알 수 있습니다.
category_updates.rename(columns={'category': 'category_id'}, inplace=True)

# 5. 업데이트를 위해 두 데이터프레임의 인덱스를 'store_id'로 설정
stores_indexed = stores.set_index('store_id')
category_updates_indexed = category_updates.set_index('store_id')

# 6. stores 데이터의 'category_id' 열을 'reviews'의 category 정보로 업데이트
stores_indexed.update(category_updates_indexed)

# 7. 인덱스를 다시 일반 열로 되돌립니다.
updated_stores = stores_indexed.reset_index()

# 8. 결과를 새로운 CSV 파일로 저장
updated_stores.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"'{reviews_path}'의 'category' 정보로 '{store_path}'의 'category_id'를 업데이트하여 '{output_path}'에 저장했습니다.")

# (선택) 얼마나 많은 가게의 카테고리가 변경되었는지 확인
try:
    comparison_df = pd.merge(stores, updated_stores, on='store_id', suffixes=('_before', '_after'))
    changed_rows = comparison_df[comparison_df['category_id_before'] != comparison_df['category_id_after']]
    num_changed = len(changed_rows)

    print(f"\n총 {num_changed}개 가게의 카테고리가 변경되었습니다.")
    if num_changed > 0:
        print("\n--- 변경 내역 예시 (최대 5개) ---")
        print(changed_rows[['store_id', 'category_id_before', 'category_id_after']].head())
except Exception as e:
    print(f"\n변경 내역 확인 중 오류 발생: {e}")

'../data/raw/reviews2.csv'의 'category' 정보로 '../data/raw/ymj2.csv'의 'category_id'를 업데이트하여 '../data/raw/ymj3.csv'에 저장했습니다.

총 125개 가게의 카테고리가 변경되었습니다.

--- 변경 내역 예시 (최대 5개) ---
   store_id  category_id_before  category_id_after
0       194                12.0                2.0
1       505                 NaN               11.0
2       627                 NaN                6.0
3       892                 NaN                3.0
4       991                 NaN                2.0


In [8]:
import pandas as pd

# CSV 파일 읽기
df = pd.read_csv("../data/raw/ymj3_final.csv")

# 'category' 컬럼이 존재하면 제거
if 'category' in df.columns:
    df = df.drop(columns=['category'])
else:
    print("경고: 'category' 컬럼이 존재하지 않습니다.")

# 같은 이름으로 덮어쓰기
df.to_csv("../data/raw/ymj3_final.csv", index=False)

print("컬럼 제거 및 저장 완료: ymj3_final.csv")

경고: 'category' 컬럼이 존재하지 않습니다.
컬럼 제거 및 저장 완료: ymj3_final.csv
